In [1]:
# load libaries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as sm

The tool outputs are created by the script "merge_tool_outputs.ipynb".\
The tools have max capacity and therefore the data set with antibodies had to be split up for predictions to be performed. To reduce risk of intefering with the merging of the files, to make each script shorter and easier to understand and to be able to remove the originals files this computation of scores part and the merge tool outputs part have been separated. 

In [2]:
# Load merged tool outputs
netMHC1_EL_pep9 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC1_EL_pep9_only_useful_scores.csv")
netMHC1_EL_pep14 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC1_EL_pep14_only_useful_scores.csv")
netMHC_II_EL_pep15 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC_II_EL_pep15_only_useful_scores.csv")
netMHC_II_EL_pep12 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC_II_EL_pep12_only_useful_scores.csv")
waltz = pd.read_csv("merged_tool_outputs/only_useful_scores/waltz_merged_tool_outputs.csv")
biophi_relaxed = pd.read_csv("merged_tool_outputs/only_useful_scores/biophi_relaxed_only_useful_scores.csv")
biophi_strict = pd.read_csv("merged_tool_outputs/only_useful_scores/biophi_strict_only_useful_scores.csv")


# Compute scores
netMHCpan tools give one score for each peptide - HLA allele combination
Therefore, for each antibody a immunogenicity score will be calculated. The definition of what score is considerd immunogenetic differs for the different tools \
for each netMHC score of interest: precentile score, immunogenicity socre and pre processing score

In [3]:
# netMHC1_EL_pep9 percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep9_percentile = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_percentile')
    )

# netMHC1_EL_pep9 Immunogenicity score 

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
class1pMHC_immunogen_pep9 = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='class1pMHC_immunogen_pep9')
    )

# netMHC1_EL_pep9 Preprocessing score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
basicPreProcessMCH1_pep9 = netMHC1_EL_pep9.groupby('antibody')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'basicPreProcessMCH1_pep9'})

# netMHC1_EL_pep14

# Percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep14_percentile = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_percentile')
    )

# Immunogenicity score

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
class1pMHC_immunogen_pep14 = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='class1pMHC_immunogen_pep14')
    )

# Pre-proocessing score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 

basicPreProcessMCH1_pep14 = netMHC1_EL_pep14.groupby('antibody')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'basicPreProcessMCH1_pep14'})


# netMHC_II_EL_pep12

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep12_percentile = (
    netMHC_II_EL_pep12.assign(immunogenic=netMHC_II_EL_pep12['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep12_percentile')
    )

# Immunogenicity score

# remove the rows with the immunogenicity score of '-' before calculating the mean
netMHC_II_EL_pep12 = netMHC_II_EL_pep12[netMHC_II_EL_pep12['immunogenicity score'] != '-']
# make the column with the immunogenicity score into a numeric column
netMHC_II_EL_pep12['immunogenicity score'] = pd.to_numeric(netMHC_II_EL_pep12['immunogenicity score'])

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody.
CD4Episcore_pep12 = netMHC_II_EL_pep12.groupby('antibody')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'CD4Episcore_pep12'})


# Does not have pre processing score. Due to tool not being able to predict pre processing on peptides shorter than 13 amino acids


# netMHC_II_EL_pep15

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep15_percentile = (
    netMHC_II_EL_pep15.assign(immunogenic=netMHC_II_EL_pep15['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep15_percentile')
    )

# Immunogenicity score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
CD4Episcore_pep15 = netMHC_II_EL_pep15.groupby('antibody')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'CD4Episcore_pep15'})

# Pre-proocessing score
# MHC class 2 has 2 preprocessing scores of interest: mhcii-np cleavage probability score and mhcii-np cleavage probability percentile rank

# mhcii-np cleavage probability

# remove the rows with the cleavage probability score of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability score'] != '-']
# make the column with the cleavage probability score into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability score'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability score'])
# Compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
MHCII_NP_pep15_cleavProb = netMHC_II_EL_pep15.groupby('antibody')['mhcii-np cleavage probability score'].mean().reset_index().rename(columns={'mhcii-np cleavage probability score': 'MHCII_NP_pep15_cleavProb'})

# mhcii-np cleavage probability percentile rank

# remove the rows with the cleavage probability percentile rank of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] != '-']
# make the column with the cleavage probability percentile rank into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'])
# compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
MHCII_NP_pep15_cleavProbPercentile = netMHC_II_EL_pep15.groupby('antibody')['mhcii-np cleavage probability percentile rank'].mean().reset_index().rename(columns={'mhcii-np cleavage probability percentile rank': 'MHCII_NP_pep15_cleavProbPercentile'})


# waltz

# rename the column '...' to nr_aggs
waltz = waltz.rename(columns={'...': 'nr_aggs'})

# compute the total number of amio acids that are considerd immunogenetic
def sum_ranges(s):
    if pd.isna(s):
        return 0
    total = 0
    for part in s.split(';'):
        start, end = part.strip().split('-')
        total += int(end) - int(start) + 1 # beacuse the values are inclusive
    return total

waltz['nr_aggs'] = waltz['waltz_score'].apply(sum_ranges)

# biophi 
# only rename the antibody column so it matches the rest
biophi_relaxed = biophi_relaxed.rename(columns={'Antibody': 'antibody'})
biophi_strict = biophi_strict.rename(columns={'Antibody': 'antibody'})

# Load ADA scores + add one previously predicted antibody & nanobodies


In [4]:

ADA = pd.read_csv("ADA_Clinical_Ab_2021_biophiDataset.csv")
# extract only the ADA (here called Immunogenicity) and Antibody (name) columns
ADA = ADA[["Antibody", "Immunogenicity"]].copy()
# make all antibody names lowercase
ADA["Antibody"] = ADA["Antibody"].str.lower()
# rename antibody and immunogenicity column
ADA = ADA.rename(columns={'Antibody': 'antibody', 'Immunogenicity': 'ADA_percentage'})


# load another dataset with antibodies, that have been predicited previously
previous_37AB = pd.read_csv("../Antibodies/ADA_combined_features.csv")
# make all antibody names lowercase
previous_37AB["antibody"] = previous_37AB["antibody"].str.lower()
# check which antibodies in previous_37AB does not exist in ADA
antibodies_not_in_ADA = set(previous_37AB['antibody']) - set(ADA['antibody'])
print("Antibodies in previous_37AB that are not in ADA:", antibodies_not_in_ADA)    
# Will add this antibody in the merging step

# load dataset with nanobodies, that have been predicted previously
previous_7NB = pd.read_csv("../Nanobodies/all_predictors_NB.csv")
# make all nanobody names lowercase
previous_7NB["antibody"] = previous_7NB["antibody"].str.lower()
# check which antibodies in previous_37AB does not exist in ADA
nanobodies_not_in_ADA = set(previous_7NB['antibody']) - set(ADA['antibody'])
print("Antibodies in previous_37AB that are not in ADA:", nanobodies_not_in_ADA)  
# Will add these nanobodies in the merging step


Antibodies in previous_37AB that are not in ADA: {'visilizumab'}
Antibodies in previous_37AB that are not in ADA: {'caplacizumab', 'gontivimab_alx-0171', 'ozoralizumab', 'vobarilizumab', '2rs15d', 'tarperprumig_alxn1820'}


# Merge scores from all predictors into one df

In [5]:
MHCII_NP_pep15_cleavProbPercentile

,antibody,MHCII_NP_pep15_cleavProbPercentile
0,3f8,29.354146
1,abagovomab,24.888571
2,abciximab,26.229048
3,abrilumab,29.753171
4,actoxumab,31.544048
...,...,...
212,veltuzumab,29.664048
213,xentuzumab,32.054524
214,zalutumumab,25.100465
215,zolbetuximab,27.603953


In [6]:
# create df with all the dfs that shall me merged into one
dfs_to_merge = [
    netMHC1_pep9_percentile,
    class1pMHC_immunogen_pep9,
    basicPreProcessMCH1_pep9,
    netMHC1_pep14_percentile,
    class1pMHC_immunogen_pep14,
    basicPreProcessMCH1_pep14,
    netMHC_II_pep12_percentile,
    CD4Episcore_pep12,
    netMHC_II_pep15_percentile,
    CD4Episcore_pep15,
    MHCII_NP_pep15_cleavProb,
    MHCII_NP_pep15_cleavProbPercentile,
]

# Rename the sequence name column to antibody to make the merging easier. 
dfs_to_merge = [df.rename(columns={'sequence name': 'antibody'}) for df in dfs_to_merge]

# Create the df with all scores, start with the ADA scores
all_predictors_andADA = ADA

# Merge all dfs on antibody name
for df in dfs_to_merge:
    all_predictors_andADA = all_predictors_andADA.merge(df, on='antibody', how='left')

# remove all whitespaces in antobdy column
all_predictors_andADA['antibody'] = all_predictors_andADA['antibody'].str.strip()

# Then merge the three last dfs
all_predictors_andADA = all_predictors_andADA.merge(waltz[['antibody','nr_aggs']], on='antibody', how='left').rename(columns={'nr_aggs': 'waltz_nr_aggs'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_relaxed, on='antibody', how='left').rename(columns={'OASis Identity': 'biophi_KabKabRelaxed_score'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_strict, on='antibody', how='left').rename(columns={'OASis Identity': 'biophi_KabKabStrict_score'})

# Add the row visilizumab from the previous_37AB df to the all_predictors_andADA, they should have the same columns

# add the row with the antibody visilizumab from the previous_37AB df to the all_predictors_andADA df, 
# but only if it does not already exist in all_predictors_andADA

visilizumab_row = previous_37AB[previous_37AB['antibody'] == 'visilizumab']
all_predictors_andADA = pd.concat([all_predictors_andADA, visilizumab_row], ignore_index=True)

# Add the nanobodies from the  previous_7NB df to the all_predictors_andADA df
names = ['gontivimab_alx-0171', 'vobarilizumab', 'ozoralizumab', 'caplacizumab', '2rs15d', 'tarperprumig_alxn1820']

nanobodies_to_add = previous_7NB[
    previous_7NB['antibody'].isin(names)
]

# keep only those not already in the target df
nanobodies_to_add = nanobodies_to_add[
    ~nanobodies_to_add['antibody'].isin(all_predictors_andADA['antibody'])
]

all_predictors_andADA = pd.concat(
    [all_predictors_andADA, nanobodies_to_add],
    ignore_index=True
)


#all_predictors_andADA = all_predictors_andADA.drop(columns=['CD4Episcore_pep12','CD4Episcore_pep15'])
all_predictors_andADA

,antibody,ADA_percentage,netMHC1_pep9_percentile,class1pMHC_immunogen_pep9,basicPreProcessMCH1_pep9,netMHC1_pep14_percentile,class1pMHC_immunogen_pep14,basicPreProcessMCH1_pep14,netMHC_II_pep12_percentile,CD4Episcore_pep12,netMHC_II_pep15_percentile,CD4Episcore_pep15,MHCII_NP_pep15_cleavProb,MHCII_NP_pep15_cleavProbPercentile,waltz_nr_aggs,biophi_KabKabRelaxed_score,biophi_KabKabStrict_score
0,3f8,100.00,3.445306,37.674419,-3.373804,0.229277,38.095238,-3.531327,8.010336,99.118753,10.229277,94.757871,0.114987,29.354146,51,0.497585,0.188406
1,abagovomab,68.10,2.837241,44.495413,-3.417786,0.243436,41.314554,-3.561854,7.838071,99.484626,9.043928,95.568972,0.120447,24.888571,53,0.433333,0.180952
2,abciximab,35.50,3.227999,39.908257,-3.443564,0.208659,37.558685,-3.587375,8.354866,99.494547,9.043928,96.271372,0.119223,26.229048,23,0.428571,0.147619
3,abrilumab,0.40,2.508961,39.170507,-3.543928,0.192173,35.377358,-3.673514,7.579673,99.620457,9.130060,96.304235,0.146311,29.753171,30,0.985646,0.837321
4,actoxumab,0.00,2.966315,44.343891,-3.409749,0.240055,41.203704,-3.548555,7.996633,99.447036,7.665805,96.583658,0.095804,31.544048,39,0.924883,0.835681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,caplacizumab,11.00,3.549383,53.333333,-3.381630,0.193237,60.000000,-3.505279,11.419753,98.935075,10.466989,94.663083,0.121135,32.588182,7,0.608333,0.391667
220,2rs15d,20.00,2.596054,41.121495,-3.471023,0.108932,47.058824,-3.568047,5.291005,99.488019,4.409171,95.527548,0.137821,23.978421,25,0.392523,0.168224
221,gontivimab_alx-0171,34.10,2.944748,61.065574,-3.475876,0.247947,61.924686,-3.606141,9.750567,99.351057,7.716049,95.516623,0.155020,23.435319,80,0.457627,0.169492
222,vobarilizumab,41.00,2.842755,41.228070,-3.435665,0.298954,34.080717,-3.571817,7.160494,99.149196,6.748971,95.134060,0.151941,22.027045,38,0.752667,0.550037


In [7]:
# Make all white space and '-' in the column names to '_'
all_predictors_andADA.columns = (
    all_predictors_andADA.columns
    .str.replace(r'[\s\-]+', '_', regex=True)
)

In [8]:
# export score table
all_predictors_andADA.to_csv("all_predictors_224AB(biophidata).csv", index=False)